# Knowledge distillation — small student, big teacher

**Model compression** (applies to every track's foundation models).

A small **student** learns from a larger **teacher**'s *soft* probabilities (richer than hard labels). On **real handwritten digits**, the distilled student — trained on the teacher's soft targets over all (unlabeled) data — nearly matches the teacher and beats an identical student trained on only a few hard labels.

> Real data: scikit-learn `load_digits` (1,797 real 8×8 images, 10 classes). CPU is fine.

In [ ]:
import os, math, torch, torch.nn as nn, torch.nn.functional as F, matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
device = "cuda" if torch.cuda.is_available() else "cpu"
STEPS = int(os.environ.get("STEPS", 1500)); torch.manual_seed(0)

## 1 · Real handwritten digits (10-class), held-out test split

In [ ]:
d = load_digits()
X = torch.tensor(d.data / 16.0, dtype=torch.float32); Y = torch.tensor(d.target, dtype=torch.long)
Xtr, Xte, Ytr, Yte = train_test_split(X, Y, test_size=0.3, stratify=Y, random_state=0)
Xtr, Xte, Ytr, Yte = Xtr.to(device), Xte.to(device), Ytr.to(device), Yte.to(device)
print("train", tuple(Xtr.shape), "test", tuple(Xte.shape), "classes", int(Y.max()) + 1)

## 2 · Train a big teacher on the full labelled training set

In [ ]:
teacher = nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 10)).to(device)
o = torch.optim.Adam(teacher.parameters(), 2e-3)
for _ in range(800): o.zero_grad(); F.cross_entropy(teacher(Xtr), Ytr).backward(); o.step()
teacher_acc = (teacher(Xte).argmax(1) == Yte).float().mean().item(); print(f"teacher test acc {teacher_acc:.3f}")

## 3 · A tiny student: a few hard labels vs. distillation over all data
The student only sees **100 labels**. Distillation instead trains it on the teacher's *soft* predictions across **all** training data — transferring "dark knowledge" the hard labels don't carry.

In [ ]:
def small(): return nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10)).to(device)
nlab = 100; Xs, Ys = Xtr[:nlab], Ytr[:nlab]
with torch.no_grad(): soft = F.softmax(teacher(Xtr) / 4.0, 1)        # teacher's dark knowledge over ALL data
def train_student(distill):
    s = small(); o = torch.optim.Adam(s.parameters(), 3e-3); h = []
    for step in range(STEPS + 1):
        loss = (16.0 * F.kl_div(F.log_softmax(s(Xtr) / 4.0, 1), soft, reduction="batchmean")
                if distill else F.cross_entropy(s(Xs), Ys))
        o.zero_grad(); loss.backward(); o.step()
        if step % max(1, STEPS // 10) == 0: h.append((step, (s(Xte).argmax(1) == Yte).float().mean().item()))
    return s, h
_, h_plain = train_student(False); _, h_distill = train_student(True)
print(f"student test acc — {nlab} labels {h_plain[-1][1]:.3f}  vs  distilled (teacher over all data) {h_distill[-1][1]:.3f}")

## 4 · Compare

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(*zip(*h_plain), label=f"{nlab} labels"); ax.plot(*zip(*h_distill), label="distilled (all data)")
ax.axhline(teacher_acc, ls="--", c="C7", label="teacher"); ax.set_xlabel("step"); ax.set_ylabel("student test acc"); ax.legend(); ax.grid(alpha=.3)
plt.show()

## Save artifacts — checkpoint · metrics · figure
Everything is written to **outputs/LM_distillation/** — the model checkpoint, the full loss/eval history as JSON, and the final figure — then zipped. Colab sessions are ephemeral, so the last lines show how to download the zip or copy it to Google Drive.

In [ ]:
import os, json, torch, shutil
run = "outputs/LM_distillation"; os.makedirs(run, exist_ok=True)
torch.save(teacher.state_dict(), f"{run}/teacher.pt")
json.dump({"teacher": teacher_acc, "student_plain": h_plain, "student_distill": h_distill}, open(f"{run}/metrics.json", "w"), indent=2)
try:
    fig.savefig(f"{run}/figure.png", dpi=120, bbox_inches="tight")
except Exception: pass
shutil.make_archive(run, "zip", run)
print("saved to", run, "->", sorted(os.listdir(run)))
# keep it past the session:  from google.colab import files; files.download(f"{run}.zip")
# or mount Drive:  from google.colab import drive; drive.mount('/content/drive')  # then shutil.copytree(run, "/content/drive/MyDrive/"+run)

## (Optional) Publish to the Hugging Face Hub
Upload this run as a **model repo** — the checkpoint, `metrics.json` (full loss/eval history) and the results figure, embedded in an auto-generated model card. Do it for each lab, then group them into a **Collection** on your HF profile (Profile → New collection), or with the commented `add_collection_item` call below. It needs a **write token**, so it only runs once you sign in.

In [ ]:
# (optional) publish this run as a Hugging Face model repo — checkpoint + metrics + plot
!pip -q install huggingface_hub
from huggingface_hub import HfApi, notebook_login
import os
notebook_login()   # paste a WRITE token from https://huggingface.co/settings/tokens
api = HfApi(); user = api.whoami()["name"]
lab = os.path.basename(run); repo_id = f"{user}/ropedia-" + lab.lower().replace("_", "-")
fig = "\n![results](figure.png)\n" if os.path.exists(f"{run}/figure.png") else ""
open(f"{run}/README.md", "w").write("---\ntags: [ropedia-academy, education]\n---\n# " + lab + "\n\nTrained in **Ropedia Academy** (educational lab). Checkpoint, full loss/eval history (metrics.json) and the results figure are included." + fig)
api.create_repo(repo_id, repo_type="model", exist_ok=True)
api.upload_folder(folder_path=run, repo_id=repo_id, commit_message="Upload from Ropedia Academy")
print("uploaded ->", "https://huggingface.co/" + repo_id)
# group everything into one Collection (run once, after a few uploads):
# from huggingface_hub import create_collection, add_collection_item
# col = create_collection("Ropedia Academy - trained models", namespace=user, exists_ok=True)
# add_collection_item(col.slug, item_id=repo_id, item_type="model")

## How this links to tracks A–D
Compresses the foundation models from **A / B / C / D** for deployment.

### Where to go next
- Distill a large fine-tuned LLM into a small one for cheap deployment; combine with **quantization** (the serving lab).
- Feature/attention distillation transfers more than just logits.